# 健康 vs 患者 MSN 网络边差异分析 (NBS)

本 notebook 仅做 **健康对照 vs 患者** 的 Network-Based Statistic 分析：

- 从 `results/health_msn_matrices/` 与 `results/patient_msn_matrices/` 加载 MSN 矩阵。
- 协变量来自 `data/health_info.csv` 与 `data/patient_info.csv`（ID, Age, Gender）。
- 设计矩阵：`[1, group, sex, age_centered, global_centered]`，对比向量 `[0, 1, 0, 0, 0]`。
- 调用 `nbs_glm_from_matrices` 做 GLM-NBS，结果保存至 `results/nbs/`。

In [ ]:
# 环境与路径（与 run_msn_pipeline / health_msn 一致）

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "msn_pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print("项目根:", ROOT)

In [ ]:
import numpy as np
import pandas as pd

from msn_stats.pipelines import run_nc_vs_patient_nbs
from msn_stats.data import N_NODES

DATA_ROOT = ROOT / "data"
RESULTS_ROOT = ROOT / "results"
HEALTH_INFO_PATH = DATA_ROOT / "health_info.csv"
PATIENT_INFO_PATH = DATA_ROOT / "patient_info.csv"
HEALTH_MSN_DIR = RESULTS_ROOT / "health_msn_matrices"
PATIENT_MSN_DIR = RESULTS_ROOT / "patient_msn_matrices"
NBS_OUT_DIR = RESULTS_ROOT / "nbs"
NBS_OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. 加载协变量与 MSN 矩阵

- 只保留在 info 表中有记录、且存在对应 MSN 文件的被试。
- Gender：f→0, m→1。

In [ ]:
def load_msn_matrix(csv_file: Path, n_roi: int = 148) -> np.ndarray:
    """读取 MSN CSV，提取 148×148 数值矩阵（跳过首行首列标签）。"""
    df = pd.read_csv(csv_file, header=None)
    return df.iloc[1 : n_roi + 1, 1 : n_roi + 1].values.astype(np.float64)


# 健康人：info + MSN 文件（{id}_feature_matrix_msn.csv）
health_info = pd.read_csv(HEALTH_INFO_PATH, dtype={"ID": str, "Age": np.float64})
health_info["sex"] = health_info["Gender"].map({"f": 0, "m": 1})

health_ids = []
health_matrices = []
health_ages = []
health_sexes = []
for _, row in health_info.iterrows():
    sid = str(row["ID"]).strip()
    f = HEALTH_MSN_DIR / f"{sid}_feature_matrix_msn.csv"
    if not f.exists():
        continue
    health_ids.append(sid)
    health_matrices.append(load_msn_matrix(f, N_NODES))
    health_ages.append(row["Age"])
    health_sexes.append(row["sex"])

if health_matrices:
    health_matrices = np.stack(health_matrices, axis=0)
    health_ages = np.asarray(health_ages, dtype=np.float64)
    health_sexes = np.asarray(health_sexes, dtype=np.float64)
else:
    health_matrices = np.empty((0, N_NODES, N_NODES), dtype=np.float64)
    health_ages = np.array([], dtype=np.float64)
    health_sexes = np.array([], dtype=np.float64)

n_health = len(health_ids)
print(f"健康人: {n_health} 人，矩阵形状 {health_matrices.shape}")

In [ ]:
# 患者：info + MSN 文件（{id}_msn_matrix.csv）；ID 统一当作字符串，与文件名完全一致匹配
patient_info = pd.read_csv(PATIENT_INFO_PATH, dtype={"ID": str, "Age": np.float64})
patient_info["sex"] = patient_info["Gender"].map({"f": 0, "m": 1})

patient_ids = []
patient_matrices = []
patient_ages = []
patient_sexes = []
for _, row in patient_info.iterrows():
    sid = str(row["ID"]).strip()
    f = PATIENT_MSN_DIR / f"{sid}_msn_matrix.csv"
    if not f.exists():
        continue
    patient_ids.append(sid)
    patient_matrices.append(load_msn_matrix(f, N_NODES))
    patient_ages.append(row["Age"])
    patient_sexes.append(row["sex"])

if patient_matrices:
    patient_matrices = np.stack(patient_matrices, axis=0)
    patient_ages = np.asarray(patient_ages, dtype=np.float64)
    patient_sexes = np.asarray(patient_sexes, dtype=np.float64)
else:
    patient_matrices = np.empty((0, N_NODES, N_NODES), dtype=np.float64)
    patient_ages = np.array([], dtype=np.float64)
    patient_sexes = np.array([], dtype=np.float64)

n_patient = len(patient_ids)
print(f"患者: {n_patient} 人，矩阵形状 {patient_matrices.shape}")

## 2. （可选）Global shift 检查

若整体连接强度在组间差异显著，建议在 GLM 中保留 global_centered；下方已纳入。

In [ ]:
from scipy import stats

g_health = global_strength[group == 0]
g_patient = global_strength[group == 1]
if g_health.size > 0 and g_patient.size > 0:
    stat, p_global = stats.ttest_ind(g_health, g_patient)
    print("Global strength 健康 vs 患者: stat = %.4f, p = %.6f" % (stat, p_global))
    if p_global < 0.05:
        print("→ 组间差异显著，设计矩阵中已包含 global_centered。")
else:
    print("任一组无被试，跳过 global shift 检查。")

In [ ]:
# 合并为全样本：健康在前，患者在后
matrices_all = np.concatenate([health_matrices, patient_matrices], axis=0)
n_all = matrices_all.shape[0]
group = np.concatenate([np.zeros(n_health, dtype=np.float64), np.ones(n_patient, dtype=np.float64)])
sex_all = np.concatenate([health_sexes, patient_sexes])
age_all = np.concatenate([health_ages, patient_ages])

# 缺失 Age/Gender 的插补：用全样本均值（若存在缺失）
for arr in (sex_all, age_all):
    mask = np.isnan(arr)
    if mask.any():
        arr[mask] = np.nanmean(arr[~mask])

# Global strength：每人 MSN 矩阵全体元素均值，再中心化
global_strength = matrices_all.reshape(n_all, -1).mean(axis=1).astype(np.float64)
global_centered = global_strength - global_strength.mean()
age_centered = age_all - age_all.mean()

# 设计矩阵：[1, group, sex, age_centered, global_centered]
design = np.column_stack([
    np.ones(n_all, dtype=np.float64),
    group,
    sex_all,
    age_centered,
    global_centered,
])
contrast = np.array([0.0, 1.0, 0.0, 0.0, 0.0], dtype=np.float64)

print("合并后被试数:", n_all)
print("Design shape:", design.shape)
print("Contrast (group 主效应):", contrast)

## 3. NBS 分析

主阈值与置换次数与 `分析网络边差异.ipynb` 中健康 vs 患者部分一致。

In [ ]:
# 诊断：观测 t 图分布 —— 若 threshold=6 仍得到 10878 条边，说明几乎所有边的 |t| 都 >= 6
if res_nbs is not None:
    t = res_nbs.t_map
    n_edges = len(t)
    print("t 统计量: min = %.3f, max = %.3f, mean(|t|) = %.3f" % (t.min(), t.max(), np.abs(t).mean()))
    print("|t| >= 4 的边数: %d / %d (%.1f%%)" % ((np.abs(t) >= 4).sum(), n_edges, 100 * (np.abs(t) >= 4).sum() / n_edges))
    print("|t| >= 6 的边数: %d / %d (%.1f%%)" % ((np.abs(t) >= 6).sum(), n_edges, 100 * (np.abs(t) >= 6).sum() / n_edges))
    if (np.abs(t) >= 6).sum() == n_edges:
        print("→ 所有边的 |t| 都 >= 6，说明组效应极大，很可能是 design/数据顺序或尺度有问题，建议检查。")

In [ ]:
primary_threshold = 6
n_perm = 5000
nbs_seed = 123

res_dict = run_nc_vs_patient_nbs(
    health_msn_dir=HEALTH_MSN_DIR,
    patient_msn_dir=PATIENT_MSN_DIR,
    health_info_path=HEALTH_INFO_PATH,
    patient_info_path=PATIENT_INFO_PATH,
    results_root=RESULTS_ROOT,
    primary_threshold=primary_threshold,
    n_perm=n_perm,
    two_tailed=True,
    seed=nbs_seed,
)

res_nbs = res_dict["nbs_result"]
print("Observed components:", res_nbs.n_components)
print("Component sizes (edges):", res_nbs.component_sizes)
print("FWER-corrected p-value(s):", res_nbs.component_pvalues)

## 4. 结果保存

将 NBS 结果写入 `results/nbs/`，便于复现与后续分析对齐。

In [ ]:
if res_nbs is not None:
    out_name = f"nbs_healthy_vs_patient_primary{int(primary_threshold)}_perm{n_perm}"
    np.save(NBS_OUT_DIR / f"{out_name}.npy", {
        "t_map": res_nbs.t_map,
        "component_masks": res_nbs.component_masks,
        "component_pvalues": res_nbs.component_pvalues,
        "component_sizes": res_nbs.component_sizes,
        "threshold": res_nbs.threshold,
        "n_perm": res_nbs.n_perm,
        "n_nodes": res_nbs.n_nodes,
    }, allow_pickle=True)
    # 显著成分的边数及 p 值摘要保存为 CSV
    summary = pd.DataFrame({
        "component": np.arange(res_nbs.n_components),
        "n_edges": res_nbs.component_sizes,
        "p_fwer": res_nbs.component_pvalues,
    })
    summary.to_csv(NBS_OUT_DIR / f"{out_name}.csv", index=False)
    print("已保存:", NBS_OUT_DIR / f"{out_name}.npy", NBS_OUT_DIR / f"{out_name}.csv")
    for k, mask in enumerate(res_nbs.component_masks):
        mat = component_mask_to_matrix(mask, res_nbs.n_nodes)
        np.save(NBS_OUT_DIR / f"{out_name}_component{k}.npy", mat)
    print("各成分掩码矩阵已保存为", out_name, "_component*.npy")
else:
    print("无 NBS 结果，未写入文件。")